# Task 4 — Disparità regionali nella salute pubblica (solo UE)

Pulizia e unificazione dei dataset regionali Eurostat:
- Speranza di vita (demo_r_mlifexp)
- Tasso di mortalità standardizzato (hlth_cd_asdr2)

Output: CSV con Stato, Macroregione, Regione, pronto per MySQL / Power BI.

In [11]:
import pandas as pd
from pathlib import Path

CARTELLA_OUTPUT = Path("output")
CARTELLA_OUTPUT.mkdir(parents=True, exist_ok=True)

STATI_UE = {
    "Austria", "Belgio", "Bulgaria", "Cipro", "Cechia", "Germania",
    "Danimarca", "Estonia", "Grecia", "Spagna", "Finlandia", "Francia",
    "Croazia", "Ungheria", "Irlanda", "Italia", "Lituania", "Lussemburgo",
    "Lettonia", "Malta", "Paesi Bassi", "Polonia", "Portogallo", "Romania",
    "Svezia", "Slovenia", "Slovacchia"
}

MAPPA_MACRO_UE = {
    "Svezia": "Europa settentrionale", "Finlandia": "Europa settentrionale",
    "Danimarca": "Europa settentrionale", "Estonia": "Europa settentrionale",
    "Lettonia": "Europa settentrionale", "Lituania": "Europa settentrionale",
    "Irlanda": "Europa settentrionale",
    "Germania": "Europa occidentale", "Francia": "Europa occidentale",
    "Paesi Bassi": "Europa occidentale", "Belgio": "Europa occidentale",
    "Lussemburgo": "Europa occidentale", "Austria": "Europa occidentale",
    "Italia": "Europa meridionale", "Spagna": "Europa meridionale",
    "Portogallo": "Europa meridionale", "Grecia": "Europa meridionale",
    "Malta": "Europa meridionale", "Cipro": "Europa meridionale",
    "Croazia": "Europa meridionale",
    "Polonia": "Europa orientale", "Cechia": "Europa orientale",
    "Slovacchia": "Europa orientale", "Ungheria": "Europa orientale",
    "Romania": "Europa orientale", "Bulgaria": "Europa orientale",
    "Slovenia": "Europa orientale",
}

MAPPA_STATI = {
    "AT": "Austria", "BE": "Belgio", "BG": "Bulgaria", "CY": "Cipro",
    "CZ": "Cechia", "DE": "Germania", "DK": "Danimarca", "EE": "Estonia",
    "EL": "Grecia", "ES": "Spagna", "FI": "Finlandia", "FR": "Francia",
    "HR": "Croazia", "HU": "Ungheria", "IE": "Irlanda", "IT": "Italia",
    "LT": "Lituania", "LU": "Lussemburgo", "LV": "Lettonia", "MT": "Malta",
    "NL": "Paesi Bassi", "PL": "Polonia", "PT": "Portogallo", "RO": "Romania",
    "SE": "Svezia", "SI": "Slovenia", "SK": "Slovacchia"
}

def estrai_codice_nome(val):
    """Estrae codice e nome in modo sicuro da un singolo valore."""
    if pd.isna(val):
        return "", ""
    s = str(val)
    if ":" in s:
        parti = s.split(":", 1)
        return parti[0].strip(), parti[1].strip()
    return s.strip(), s.strip()

In [12]:
# ============================================================
# 1. SPERANZA DI VITA (NUTS 2)
# ============================================================
df_le = pd.read_csv("data/T4.1_demo_r_mlifexp__custom_24078981_linear_2_0.csv")
df_le.columns = df_le.columns.str.strip()

colonne_le = [
    "sex: Sex",
    "age: Age class",
    "geo: Geopolitical entity (reporting)",
    "TIME_PERIOD: Time",
    "OBS_VALUE: Observation value"
]
df_le = df_le[colonne_le].copy()

df_le = df_le.rename(columns={
    "sex: Sex": "Genere",
    "age: Age class": "Età",
    "geo: Geopolitical entity (reporting)": "Geo_raw",
    "TIME_PERIOD: Time": "Anno",
    "OBS_VALUE: Observation value": "Valore"
})

estratti = df_le["Geo_raw"].apply(estrai_codice_nome)
df_le["Codice_regione"] = estratti.apply(lambda x: x[0])
df_le["Nome_regione"] = estratti.apply(lambda x: x[1])

df_le = df_le[df_le["Età"].astype(str).str.contains("Less than 1 year|Y_LT1", na=False)]
df_le = df_le[df_le["Codice_regione"].str.len() == 4]

df_le["Indicatore"] = "Speranza di vita alla nascita"
df_le["Fonte"] = "demo_r_mlifexp"
df_le = df_le.dropna(subset=["Valore"])

print(f"Speranza di vita: {len(df_le)} righe")

Speranza di vita: 26596 righe


In [13]:
# ============================================================
# 2. TASSO DI MORTALITÀ STANDARDIZZATO (NUTS 2)
# ============================================================
df_mort = pd.read_csv("data/T4.2_hlth_cd_asdr2__custom_24079748_linear_2_0.csv")
df_mort.columns = df_mort.columns.str.strip()

colonne_mort = [
    "Sex",
    "Age class",
    "Geopolitical entity (reporting)",
    "TIME_PERIOD",
    "OBS_VALUE",
    "International Statistical Classification of Diseases and Related Health Problems (ICD-10)"
]
df_mort = df_mort[colonne_mort].copy()

df_mort = df_mort.rename(columns={
    "Sex": "Genere",
    "Age class": "Età",
    "Geopolitical entity (reporting)": "Geo_raw",
    "TIME_PERIOD": "Anno",
    "OBS_VALUE": "Valore",
    "International Statistical Classification of Diseases and Related Health Problems (ICD-10)": "Causa"
})

estratti = df_mort["Geo_raw"].apply(estrai_codice_nome)
df_mort["Codice_regione"] = estratti.apply(lambda x: x[0])
df_mort["Nome_regione"] = estratti.apply(lambda x: x[1])

df_mort = df_mort[df_mort["Età"].astype(str) == "Total"]
df_mort = df_mort[df_mort["Causa"].astype(str).str.contains("All causes", na=False)]
df_mort = df_mort[df_mort["Codice_regione"].str.len() == 4]

df_mort["Indicatore"] = "Tasso mortalità standardizzato"
df_mort["Fonte"] = "hlth_cd_asdr2"
df_mort = df_mort.dropna(subset=["Valore"])

print(f"Mortalità: {len(df_mort)} righe")

Mortalità: 48 righe


In [14]:
# ============================================================
# 3. UNIONE + FILTRO UE + MACROREGIONE
# ============================================================
df_t4 = pd.concat([
    df_le[["Codice_regione", "Nome_regione", "Anno", "Genere", "Indicatore", "Valore", "Fonte"]],
    df_mort[["Codice_regione", "Nome_regione", "Anno", "Genere", "Indicatore", "Valore", "Fonte"]]
], ignore_index=True)

df_t4["Codice_stato"] = df_t4["Codice_regione"].str[:2]
df_t4["Stato"] = df_t4["Codice_stato"].map(MAPPA_STATI)

df_t4 = df_t4[df_t4["Stato"].isin(STATI_UE)].copy()
df_t4["Macroregione"] = df_t4["Stato"].map(MAPPA_MACRO_UE)

df_t4["Genere"] = df_t4["Genere"].replace({
    "Females": "Femmine", "Males": "Maschi", "Total": "Totale",
    "F: Females": "Femmine", "M: Males": "Maschi", "T: Total": "Totale"
})

df_t4 = df_t4.rename(columns={
    "Codice_regione": "Codice Regione",
    "Nome_regione": "Nome Regione"
})

df_t4 = df_t4[[
    "Stato", "Macroregione", "Codice Regione", "Nome Regione",
    "Anno", "Genere", "Indicatore", "Valore", "Fonte"
]]

print(f"Tabella T4 (solo UE): {len(df_t4)} righe")
print(f"Stati: {df_t4['Stato'].nunique()} | Macroregioni: {df_t4['Macroregione'].nunique()}")
display(df_t4)

Tabella T4 (solo UE): 21495 righe
Stati: 27 | Macroregioni: 4


,Stato,Macroregione,Codice Regione,Nome Regione,Anno,Genere,Indicatore,Valore,Fonte
9,Austria,Europa occidentale,AT11,Burgenland,1990,Femmine,Speranza di vita alla nascita,79.3,demo_r_mlifexp
10,Austria,Europa occidentale,AT11,Burgenland,1991,Femmine,Speranza di vita alla nascita,78.7,demo_r_mlifexp
11,Austria,Europa occidentale,AT11,Burgenland,1992,Femmine,Speranza di vita alla nascita,79.0,demo_r_mlifexp
12,Austria,Europa occidentale,AT11,Burgenland,1993,Femmine,Speranza di vita alla nascita,79.6,demo_r_mlifexp
13,Austria,Europa occidentale,AT11,Burgenland,1994,Femmine,Speranza di vita alla nascita,80.0,demo_r_mlifexp
...,...,...,...,...,...,...,...,...,...
25540,Slovacchia,Europa orientale,SK04,Východné Slovensko,2020,Totale,Speranza di vita alla nascita,76.6,demo_r_mlifexp
25541,Slovacchia,Europa orientale,SK04,Východné Slovensko,2021,Totale,Speranza di vita alla nascita,73.9,demo_r_mlifexp
25542,Slovacchia,Europa orientale,SK04,Východné Slovensko,2022,Totale,Speranza di vita alla nascita,76.4,demo_r_mlifexp
25543,Slovacchia,Europa orientale,SK04,Východné Slovensko,2023,Totale,Speranza di vita alla nascita,77.7,demo_r_mlifexp


In [15]:
# ============================================================
# 4. EXPORT
# ============================================================
df_t4.to_csv(CARTELLA_OUTPUT / "t4_disparita_regionali_pulito.csv", index=False)
print(f"Esportato: t4_disparita_regionali_pulito.csv ({len(df_t4)} righe)")

Esportato: t4_disparita_regionali_pulito.csv (21495 righe)
